In [1]:
print("path")

path


Got it — then the tool should do only one job:

**take a full page and help you tune a preprocessing pipeline that gives DocTR the cleanest possible detector input**, without drawing boxes or doing contour logic.

Also, your own recognition pipeline later converts crops to grayscale, resizes with bilinear interpolation, and normalizes to `[0,1]`, both in training and inference, so overly harsh binarization is not well aligned with the rest of your stack.  

# What to optimize for

For **DocTR OCD**, the preprocessing should aim to:

* reduce uneven background
* improve local text contrast
* preserve faint handwriting
* preserve page structure
* avoid turning text into broken specks
* avoid heavy line removal at this stage

# What to avoid

For detection, these usually hurt:

* hard global thresholding
* very strong sharpening
* aggressive denoising
* line removal
* morphology that merges nearby words/lines too much

---

# Best tool for your case

This notebook tool:

* preprocesses the **whole page only**
* shows multiple stages side by side
* lets you choose which stage is your final detector input
* lets you save the current best output
* lets you batch-process a folder later with the chosen parameters

---

# Notebook code

## Cell 1 — Imports and setup

```python
```

---

## Cell 2 — Preprocessing pipeline only

```python
```

---

## Cell 3 — Preview tool

```python
```

---

## Cell 4 — Interactive controls

```python
```

---

## Cell 5 — Save the currently selected output

```python
def save_current_final_output(folder="doctr_preprocessed", prefix="page"):
    global CURRENT_RESULTS, CURRENT_PARAMS

    if not CURRENT_RESULTS:
        raise RuntimeError("No processed result found yet. Move the sliders once first.")

    os.makedirs(folder, exist_ok=True)

    final_name = CURRENT_PARAMS["final_output"]
    final_img = CURRENT_RESULTS["outputs"][final_name]

    out_path = os.path.join(folder, f"{prefix}_{final_name}.png")
    cv2.imwrite(out_path, final_img)

    params_path = os.path.join(folder, f"{prefix}_{final_name}_params.json")
    with open(params_path, "w", encoding="utf-8") as f:
        json.dump(CURRENT_PARAMS, f, indent=2)

    print(f"Saved image:  {os.path.abspath(out_path)}")
    print(f"Saved params: {os.path.abspath(params_path)}")
```

Use it like:

```python
save_current_final_output(folder="doctr_preprocessed", prefix="kaithi_page_01")
```

---

## Cell 6 — Batch process a whole folder using the current settings

```python
def batch_preprocess_folder(
    input_folder,
    output_folder="doctr_preprocessed_batch",
    exts=(".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"),
):
    global CURRENT_PARAMS

    if not CURRENT_PARAMS:
        raise RuntimeError("No parameters found yet. Tune the notebook first.")

    os.makedirs(output_folder, exist_ok=True)

    params = CURRENT_PARAMS.copy()
    final_output = params.pop("final_output")

    files = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith(exts)
    ]
    files.sort()

    for fname in files:
        in_path = os.path.join(input_folder, fname)
        img = cv2.imread(in_path)
        if img is None:
            print(f"Skipped unreadable file: {fname}")
            continue

        res = preprocess_page_for_doctr(img, **params)
        out = res["outputs"][final_output]

        stem = os.path.splitext(fname)[0]
        out_path = os.path.join(output_folder, f"{stem}_{final_output}.png")
        cv2.imwrite(out_path, out)

    with open(os.path.join(output_folder, "used_params.json"), "w", encoding="utf-8") as f:
        json.dump(CURRENT_PARAMS, f, indent=2)

    print(f"Saved {len(files)} processed files to: {os.path.abspath(output_folder)}")
```

---

# Best starting preset for DocTR OCD

For your kind of faint archival table page, start with:

```python
resize_factor = 1.0

do_bg_normalization = True
bg_blur_ksize = 81

do_clahe = True
clahe_clip = 1.8
clahe_grid = 8

denoise_h = 3
median_ksize = 3

sharpen_amount = 0.1
sharpen_sigma = 1.0

threshold_method = "none"
morph_open = 0
morph_close = 0

final_output = "bg_norm"
```

Then test these final outputs in order:

1. `bg_norm`
2. `enhanced`
3. `denoised`
4. `sharpened`

I would test **thresholded outputs last**, not first.

---

# Practical tuning rules

## If DocTR misses faint text

* reduce `denoise_h`
* reduce `sharpen_amount`
* keep `threshold_method="none"`
* try `final_output="enhanced"`

## If background stains confuse detection

* increase `bg_blur_ksize`
* slightly increase `denoise_h`
* keep CLAHE moderate, not too high

## If text becomes too harsh / broken

* lower `clahe_clip`
* reduce sharpening
* avoid thresholding

## If the page still looks too washed out

* use `final_output="enhanced"`
* or slightly increase `clahe_clip`

---

# My strongest recommendation

For **DocTR detection**, do not chase an “OCR-ready black/white” look.

Chase this instead:

* text is visible
* faint strokes still exist
* background is flatter
* page geometry is preserved
* lines/table structure are still natural
* nothing looks broken into dust

That usually gives a better detector input.

If you want, next I can give you a second notebook cell that takes your chosen saved output and runs it directly through your DocTR OCD pipeline so you can compare page-to-page results quickly.


In [14]:
import os
import cv2
import json
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import (
    interact,
    IntSlider,
    FloatSlider,
    Checkbox,
    Dropdown,
    fixed,
)

imgpath = r"D:\Work\multililpi\kaithi\potential-invention\image.png"

SOURCE_BGR = cv2.imread(imgpath)
if SOURCE_BGR is None:
    raise FileNotFoundError(f"Could not read image: {imgpath}")

CURRENT_RESULTS = {}
CURRENT_PARAMS = {}

def odd(x: int) -> int:
    x = int(x)
    return x if x % 2 == 1 else x + 1


In [15]:
def preprocess_page_for_doctr(
    source_bgr,
    resize_factor=1.0,

    # grayscale / background
    to_grayscale=True,
    do_bg_normalization=True,
    bg_blur_ksize=81,

    # local contrast
    do_clahe=True,
    clahe_clip=2.0,
    clahe_grid=8,

    # denoise / sharpen
    denoise_h=4,
    median_ksize=3,
    sharpen_amount=0.2,
    sharpen_sigma=1.0,

    # optional threshold branch
    threshold_method="none",   # none / adaptive_gaussian / adaptive_mean / otsu
    block_size=41,
    c_value=7,
    invert_binary=False,

    # mild morphology only if needed
    morph_open=0,
    morph_close=0,
):
    img = source_bgr.copy()

    if resize_factor != 1.0:
        new_w = max(1, int(img.shape[1] * resize_factor))
        new_h = max(1, int(img.shape[0] * resize_factor))
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_CUBIC)

    original_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    if to_grayscale:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    work = gray.copy()

    # 1) Background normalization
    bg_norm = work.copy()
    if do_bg_normalization:
        k = odd(bg_blur_ksize)
        bg = cv2.GaussianBlur(work, (k, k), 0)
        bg_norm = cv2.divide(work, bg, scale=255)
        work = bg_norm.copy()

    # 2) CLAHE
    enhanced = work.copy()
    if do_clahe:
        clahe = cv2.createCLAHE(
            clipLimit=float(clahe_clip),
            tileGridSize=(int(clahe_grid), int(clahe_grid)),
        )
        enhanced = clahe.apply(work)
        work = enhanced.copy()

    # 3) Denoise
    denoised = work.copy()
    if denoise_h > 0:
        denoised = cv2.fastNlMeansDenoising(
            work, None,
            h=int(denoise_h),
            templateWindowSize=7,
            searchWindowSize=21
        )
        work = denoised.copy()

    # 4) Median blur
    medianed = work.copy()
    if median_ksize >= 3:
        mk = odd(median_ksize)
        medianed = cv2.medianBlur(work, mk)
        work = medianed.copy()

    # 5) Mild sharpen
    sharpened = work.copy()
    if sharpen_amount > 0:
        blur = cv2.GaussianBlur(work, (0, 0), sharpen_sigma)
        sharpened = cv2.addWeighted(work, 1.0 + sharpen_amount, blur, -sharpen_amount, 0)
        work = sharpened.copy()

    # 6) Optional threshold branch
    if threshold_method == "none":
        binary = work.copy()
    elif threshold_method == "adaptive_gaussian":
        binary = cv2.adaptiveThreshold(
            work, 255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            odd(block_size),
            int(c_value),
        )
    elif threshold_method == "adaptive_mean":
        binary = cv2.adaptiveThreshold(
            work, 255,
            cv2.ADAPTIVE_THRESH_MEAN_C,
            cv2.THRESH_BINARY,
            odd(block_size),
            int(c_value),
        )
    elif threshold_method == "otsu":
        _, binary = cv2.threshold(
            work, 0, 255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )
    else:
        raise ValueError("Unknown threshold_method")

    if threshold_method != "none" and invert_binary:
        binary = 255 - binary

    # 7) Optional mild morphology
    processed = binary.copy()
    if threshold_method != "none":
        if morph_open > 0:
            ok = odd(morph_open)
            kernel = np.ones((ok, ok), np.uint8)
            processed = cv2.morphologyEx(processed, cv2.MORPH_OPEN, kernel)

        if morph_close > 0:
            ck = odd(morph_close)
            kernel = np.ones((ck, ck), np.uint8)
            processed = cv2.morphologyEx(processed, cv2.MORPH_CLOSE, kernel)

    # Candidate outputs you may want to feed to DocTR
    outputs = {
        "gray": gray,
        "bg_norm": bg_norm,
        "enhanced": enhanced,
        "denoised": denoised,
        "sharpened": sharpened,
        "processed": processed,
    }

    return {
        "original_rgb": original_rgb,
        "gray": gray,
        "bg_norm": bg_norm,
        "enhanced": enhanced,
        "denoised": denoised,
        "sharpened": sharpened,
        "processed": processed,
        "outputs": outputs,
    }


In [16]:
def preview_doctr_preprocessing(
    resize_factor=1.0,

    to_grayscale=True,
    do_bg_normalization=True,
    bg_blur_ksize=81,

    do_clahe=True,
    clahe_clip=2.0,
    clahe_grid=8,

    denoise_h=4,
    median_ksize=3,
    sharpen_amount=0.2,
    sharpen_sigma=1.0,

    threshold_method="none",
    block_size=41,
    c_value=7,
    invert_binary=False,

    morph_open=0,
    morph_close=0,

    final_output="bg_norm",
):
    global CURRENT_RESULTS, CURRENT_PARAMS

    CURRENT_PARAMS = {
        "resize_factor": resize_factor,
        "to_grayscale": to_grayscale,
        "do_bg_normalization": do_bg_normalization,
        "bg_blur_ksize": bg_blur_ksize,
        "do_clahe": do_clahe,
        "clahe_clip": clahe_clip,
        "clahe_grid": clahe_grid,
        "denoise_h": denoise_h,
        "median_ksize": median_ksize,
        "sharpen_amount": sharpen_amount,
        "sharpen_sigma": sharpen_sigma,
        "threshold_method": threshold_method,
        "block_size": block_size,
        "c_value": c_value,
        "invert_binary": invert_binary,
        "morph_open": morph_open,
        "morph_close": morph_close,
        "final_output": final_output,
    }

    CURRENT_RESULTS = preprocess_page_for_doctr(
        SOURCE_BGR,
        resize_factor=resize_factor,
        to_grayscale=to_grayscale,
        do_bg_normalization=do_bg_normalization,
        bg_blur_ksize=bg_blur_ksize,
        do_clahe=do_clahe,
        clahe_clip=clahe_clip,
        clahe_grid=clahe_grid,
        denoise_h=denoise_h,
        median_ksize=median_ksize,
        sharpen_amount=sharpen_amount,
        sharpen_sigma=sharpen_sigma,
        threshold_method=threshold_method,
        block_size=block_size,
        c_value=c_value,
        invert_binary=invert_binary,
        morph_open=morph_open,
        morph_close=morph_close,
    )

    res = CURRENT_RESULTS
    final_img = res["outputs"][final_output]

    fig, axes = plt.subplots(2, 3, figsize=(20, 11))

    axes[0, 0].imshow(res["original_rgb"])
    axes[0, 0].set_title("Original")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(res["gray"], cmap="gray")
    axes[0, 1].set_title("Gray")
    axes[0, 1].axis("off")

    axes[0, 2].imshow(res["bg_norm"], cmap="gray")
    axes[0, 2].set_title("Background-normalized")
    axes[0, 2].axis("off")

    axes[1, 0].imshow(res["enhanced"], cmap="gray")
    axes[1, 0].set_title("CLAHE / enhanced")
    axes[1, 0].axis("off")

    axes[1, 1].imshow(res["sharpened"], cmap="gray")
    axes[1, 1].set_title("Denoised + sharpened")
    axes[1, 1].axis("off")

    axes[1, 2].imshow(final_img, cmap="gray")
    axes[1, 2].set_title(f"Final for DocTR: {final_output}")
    axes[1, 2].axis("off")

    plt.tight_layout()
    plt.show()

    if threshold_method != "none":
        plt.figure(figsize=(8, 5))
        plt.imshow(res["processed"], cmap="gray")
        plt.title("Threshold / morphology branch")
        plt.axis("off")
        plt.show()


In [17]:
interact(
    preview_doctr_preprocessing,

    resize_factor=FloatSlider(value=1.0, min=0.5, max=2.0, step=0.05, description="Resize"),

    to_grayscale=Checkbox(value=True, description="Gray"),
    do_bg_normalization=Checkbox(value=True, description="BG norm"),
    bg_blur_ksize=IntSlider(value=81, min=9, max=301, step=2, description="BG blur"),

    do_clahe=Checkbox(value=True, description="CLAHE"),
    clahe_clip=FloatSlider(value=2.0, min=0.5, max=6.0, step=0.1, description="CLAHE clip"),
    clahe_grid=IntSlider(value=8, min=2, max=32, step=1, description="CLAHE grid"),

    denoise_h=IntSlider(value=4, min=0, max=25, step=1, description="Denoise"),
    median_ksize=IntSlider(value=3, min=0, max=15, step=1, description="Median"),
    sharpen_amount=FloatSlider(value=0.2, min=0.0, max=2.0, step=0.1, description="Sharpen"),
    sharpen_sigma=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="Sharp σ"),

    threshold_method=Dropdown(
        options=["none", "adaptive_gaussian", "adaptive_mean", "otsu"],
        value="none",
        description="Thresh"
    ),
    block_size=IntSlider(value=41, min=3, max=151, step=2, description="Block"),
    c_value=IntSlider(value=7, min=-20, max=20, step=1, description="C"),
    invert_binary=Checkbox(value=False, description="Invert"),

    morph_open=IntSlider(value=0, min=0, max=11, step=1, description="Open"),
    morph_close=IntSlider(value=0, min=0, max=11, step=1, description="Close"),

    final_output=Dropdown(
        options=["gray", "bg_norm", "enhanced", "denoised", "sharpened", "processed"],
        value="bg_norm",
        description="Final"
    ),
);


interactive(children=(FloatSlider(value=1.0, description='Resize', max=2.0, min=0.5, step=0.05), Checkbox(valu…